In [177]:
import numpy as np
import pandas as pd
from tinyshift.series import adi_cv, hampel_filter
# Criando um dataset sintético representando 1 SKU em 1 Loja ao longo de 20 dias
np.random.seed(42)
dates = pd.date_range("2026-01-01", periods=20, freq="D")

df = pd.DataFrame(
    {
        "date": dates,
        "sales": [
            0,
            2,
            0,
            1,
            0,
            15,
            0,
            0,
            0,
            0,
            0,
            3,
            0,
            1,
            0,
            0,
            0,
            10,
            0,
            0,
        ],  # Inclui picos e zeros
        "stock_eod": [
            10,
            8,
            8,
            7,
            0,
            0,
            0,
            5,
            5,
            0,
            0,
            10,
            10,
            9,
            9,
            0,
            0,
            15,
            15,
            15,
        ],
        "is_promo": [
            0,
            0,
            0,
            0,
            0,
            1,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            1,
            0,
            0,
        ],
        "price": [
            10,
            10,
            10,
            10,
            10,
            5,
            10,
            10,
            10,
            10,
            10,
            10,
            10,
            10,
            10,
            10,
            10,
            5,
            10,
            10,
        ],
    }
)
df

,date,sales,stock_eod,is_promo,price
0,2026-01-01,0,10,0,10
1,2026-01-02,2,8,0,10
2,2026-01-03,0,8,0,10
3,2026-01-04,1,7,0,10
4,2026-01-05,0,0,0,10
5,2026-01-06,15,0,1,5
6,2026-01-07,0,0,0,10
7,2026-01-08,0,5,0,10
8,2026-01-09,0,5,0,10
9,2026-01-10,0,0,0,10


# Etapa 1: Reconstrução da Disponibilidade (Stockout Detection)

Primeiro, identifica-se exatamente em quais dias o produto esteve indisponível. Um erro comum é olhar apenas para o Stock End of Day (EOD). Se o estoque no final do dia é 0 e a venda foi 0, ou se o estoque zerou durante o dia após vender algumas unidades, ocorreu ruptura.

In [178]:
# Reconstrução do estado de prateleira (Disponibilidade)
# Se a venda limpou o estoque restante (stock_eod == 0), a demanda ficou censurada.
df["is_stockout"] = (df["stock_eod"] == 0) | (
    (df["sales"] > 0) & (df["stock_eod"] == 0)
)

# Calcula a 'janela de exposição': dias efetivamente com estoque
df["exposure_day"] = np.where(df["is_stockout"], 0, 1)

In [179]:
df

,date,sales,stock_eod,is_promo,price,is_stockout,exposure_day
0,2026-01-01,0,10,0,10,False,1
1,2026-01-02,2,8,0,10,False,1
2,2026-01-03,0,8,0,10,False,1
3,2026-01-04,1,7,0,10,False,1
4,2026-01-05,0,0,0,10,True,0
5,2026-01-06,15,0,1,5,True,0
6,2026-01-07,0,0,0,10,True,0
7,2026-01-08,0,5,0,10,False,1
8,2026-01-09,0,5,0,10,False,1
9,2026-01-10,0,0,0,10,True,0


# Etapa 2: Remoção de Anomalias Não Repetíveis (Outliers Sanitization)

Picos atípicos causados por bugs de preço, licitações corporativas esporádicas ou compras atacadistas isoladas não devem poluir a demanda base.

Métricas baseadas em desvio padrão falham em dados intermitentes/cauda longa. Utiliza-se a Amplitude Interquartil (IQR) ou MAD (Median Absolute Deviation) calculada apenas sobre dias com vendas positivas.

In [180]:
# Isolando vendas em dias com estoque disponível
positive_sales = df.loc[(df["sales"] > 0) & (~df["is_promo"]), "sales"]

# Cálculo de limites via MAD (mais robusto que Desvio Padrão para cauda longa)
median = positive_sales.median()
mad = (positive_sales - median).abs().median()
upper_limit = median + (3 * 1.4826 * mad)  # 1.4826 é a constante de conversão para Normal

# Criando coluna de vendas ajustadas de anomalias
df["sales_no_outliers"] = df["sales"].copy().astype(float)
mask_outlier = (df["sales"] > upper_limit) & (~df["is_promo"])
df.loc[mask_outlier, "sales_no_outliers"] = upper_limit

#TODO: Filtro de hampel aqui faria sentido?

In [181]:
df

,date,sales,stock_eod,is_promo,price,is_stockout,exposure_day,sales_no_outliers
0,2026-01-01,0,10,0,10,False,1,0.0
1,2026-01-02,2,8,0,10,False,1,2.0
2,2026-01-03,0,8,0,10,False,1,0.0
3,2026-01-04,1,7,0,10,False,1,1.0
4,2026-01-05,0,0,0,10,True,0,0.0
5,2026-01-06,15,0,1,5,True,0,15.0
6,2026-01-07,0,0,0,10,True,0,0.0
7,2026-01-08,0,5,0,10,False,1,0.0
8,2026-01-09,0,5,0,10,False,1,0.0
9,2026-01-10,0,0,0,10,True,0,0.0


# Etapa 3: Despromocionalização (Baseline Extraction)

A demanda promocional não reflete o consumo orgânico do produto. É preciso isolar o efeito de elevação (uplift) da promoção para estimar qual seria a venda baseline do dia.

In [182]:
# Cálculo do Lift Médio da Promoção (Comparando dias normais com dias promocionais)
avg_normal_sales = df.loc[
    (df["sales_no_outliers"] > 0) & (~df["is_promo"]) & (~df["is_stockout"]),
    "sales_no_outliers",
].mean()
avg_promo_sales = df.loc[
    (df["sales_no_outliers"] > 0) & (df["is_promo"]) & (~df["is_stockout"]),
    "sales_no_outliers",
].mean()

# Fator de Elevação (Lift Ratio)
promo_lift = (
    avg_promo_sales / avg_normal_sales
    if avg_normal_sales > 0 and not np.isnan(avg_promo_sales)
    else 1.0
)

# Despromocionalização: Reduz o impacto da promoção para obter a demanda orgânica
df["sales_baseline"] = df["sales_no_outliers"].copy()
df.loc[df["is_promo"] == 1, "sales_baseline"] = (
    df["sales_no_outliers"] / promo_lift
)

In [183]:
df

,date,sales,stock_eod,is_promo,price,is_stockout,exposure_day,sales_no_outliers,sales_baseline
0,2026-01-01,0,10,0,10,False,1,0.0,0.000
1,2026-01-02,2,8,0,10,False,1,2.0,2.000
2,2026-01-03,0,8,0,10,False,1,0.0,0.000
3,2026-01-04,1,7,0,10,False,1,1.0,1.000
4,2026-01-05,0,0,0,10,True,0,0.0,0.000
5,2026-01-06,15,0,1,5,True,0,15.0,2.625
6,2026-01-07,0,0,0,10,True,0,0.0,0.000
7,2026-01-08,0,5,0,10,False,1,0.0,0.000
8,2026-01-09,0,5,0,10,False,1,0.0,0.000
9,2026-01-10,0,0,0,10,True,0,0.0,0.000


# Etapa 4: Imputação de Vendas Perdidas por Ruptura (Estoque Zero)

Esta é a etapa crucial onde os dados de todos os tipos de demanda são higienizados diferentemente para imputar a demanda que não foi registrada devido à falta de produto.

In [184]:
# 1. Métrica de Intermitência Local
# ADI: Intervalo Médio entre Demandas (olhando apenas histórico de prateleira disponível)
sales_dates = df.loc[
    (df["sales_baseline"] > 0) & (~df["is_stockout"]), "date"
]
if len(sales_dates) > 1:
    adi = (sales_dates.diff().dt.days).mean()
else:
    adi = 1.0

# CV²: Coeficiente de Variação Quadrado da Demanda Não-Zero
nonzero_sales = df.loc[
    (df["sales_baseline"] > 0) & (~df["is_stockout"]), "sales_baseline"
]
cv2 = (
    (nonzero_sales.std() / nonzero_sales.mean()) ** 2
    if len(nonzero_sales) > 1
    else 0.0
)

# 2. Pipeline Condicional de Imputação de Demanda Latente
df["sanitized_demand"] = df["sales_baseline"].copy()

# CONDICIONAL DE ROTA CONFORME COMPORTAMENTO DA SÉRIE
if adi < 1.32 and cv2 < 0.49:
    # --- ROTA 1: DEMANDA CONTÍNUA / ALTO GIRO (SMOOTH) ---
    # Imputação via Média Móvel Local (Janela Exclui Dias de Ruptura)
    window_mean = (
        df.loc[~df["is_stockout"], "sales_baseline"]
        .rolling(window=7, min_periods=1)
        .mean()
    )
    df.loc[df["is_stockout"], "sanitized_demand"] = window_mean

elif adi >= 1.32 and cv2 < 0.49:
    # --- ROTA 2: DEMANDA INTERMITENTE (INTERMITTENT) ---
    # NÃO substitui o zero por média diária. Em vez disso, aplica a Taxa de Demanda pelo Tempo de Exposição
    # Demanda por dia de exposição = Total Vendido / Dias Com Estoque
    exposure_rate = df.loc[~df["is_stockout"], "sales_baseline"].sum() / max(
        df["exposure_day"].sum(), 1
    )

    # Imputa a taxa ponderada de ocorrência apenas no bloco contínuo de ruptura
    df.loc[df["is_stockout"], "sanitized_demand"] = exposure_rate

else:
    # --- ROTA 3: DEMANDA ERRÁTICA / LUMPY ---
    # Usa a Mediana Histórica de Dias com Venda para não inflar a variância com picos
    median_demand = df.loc[
        (df["sales_baseline"] > 0) & (~df["is_stockout"]), "sales_baseline"
    ].median()
    # Pondera pela probabilidade histórica de haver demanda no dia
    p_demand = (df["sales_baseline"] > 0).mean()
    df.loc[df["is_stockout"], "sanitized_demand"] = median_demand * p_demand

In [185]:
df

,date,sales,stock_eod,is_promo,price,is_stockout,exposure_day,sales_no_outliers,sales_baseline,sanitized_demand
0,2026-01-01,0,10,0,10,False,1,0.0,0.000,0.000000
1,2026-01-02,2,8,0,10,False,1,2.0,2.000,2.000000
2,2026-01-03,0,8,0,10,False,1,0.0,0.000,0.000000
3,2026-01-04,1,7,0,10,False,1,1.0,1.000,1.000000
4,2026-01-05,0,0,0,10,True,0,0.0,0.000,0.673077
5,2026-01-06,15,0,1,5,True,0,15.0,2.625,0.673077
6,2026-01-07,0,0,0,10,True,0,0.0,0.000,0.673077
7,2026-01-08,0,5,0,10,False,1,0.0,0.000,0.000000
8,2026-01-09,0,5,0,10,False,1,0.0,0.000,0.000000
9,2026-01-10,0,0,0,10,True,0,0.0,0.000,0.673077


In [186]:
import numpy as np
import pandas as pd
from tinyshift.stats import StatisticalInterval

def sanitize_time_series_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    """Pipeline de Higienização de Demanda utilizando Filtro de Hampel,

    Intervalos Estatísticos Dinâmicos e Métricas ADI/CV.
    """
    df = df.copy()

    # ------------------------------------------------------------------
    # ETAPA 1: Reconstrução do Estado de Prateleira (Stockout Detection)
    # ------------------------------------------------------------------
    df["is_stockout"] = (df["stock_eod"] == 0) | (
        (df["sales"] > 0) & (df["stock_eod"] == 0)
    )
    df["exposure_day"] = np.where(df["is_stockout"], 0, 1)

    # ------------------------------------------------------------------
    # ETAPA 2: Tratamento de Outliers via Hampel Filter (Local)
    # ------------------------------------------------------------------
    # Aplica o Hampel Filter considerando a janela local
    valid_sales_mask = ~df["is_stockout"] & (df["is_promo"] == 0)
    valid_sales = df.loc[valid_sales_mask, "sales"]

    if len(valid_sales) >= 7:
        # Filtro de Hampel em janela móvel de 7 períodos
        is_hampel_outlier = hampel_filter(
            valid_sales, window_size=7, factor=3.0
        )

        # Para os pontos flagged pelo Hampel, calcula o teto usando StatisticalInterval
        clean_sales_sample = valid_sales[~is_hampel_outlier].values
        if len(clean_sales_sample) > 0:
            _, upper_bound = StatisticalInterval.compute_interval(
                clean_sales_sample, method="auto"
            )
            df["sales_no_outliers"] = df["sales"].copy()
            df.loc[
                valid_sales_mask & is_hampel_outlier, "sales_no_outliers"
            ] = upper_bound
        else:
            df["sales_no_outliers"] = df["sales"].copy()
    else:
        df["sales_no_outliers"] = df["sales"].copy()

    # ------------------------------------------------------------------
    # ETAPA 3: Despromocionalização (Baseline Extraction)
    # ------------------------------------------------------------------
    avg_normal = df.loc[
        (~df["is_stockout"]) & (df["is_promo"] == 0), "sales_no_outliers"
    ].mean()
    avg_promo = df.loc[
        (~df["is_stockout"]) & (df["is_promo"] == 1), "sales_no_outliers"
    ].mean()

    promo_lift = (
        (avg_promo / avg_normal)
        if (avg_normal > 0 and not np.isnan(avg_promo))
        else 1.0
    )

    df["sales_baseline"] = df["sales_no_outliers"].astype(float)
    df.loc[df["is_promo"] == 1, "sales_baseline"] = (
        df["sales_no_outliers"] / promo_lift
    )

    # ------------------------------------------------------------------
    # ETAPA 4: Classificação via adi_cv e Imputação da Demanda Latente
    # ------------------------------------------------------------------
    valid_baseline = df.loc[~df["is_stockout"], "sales_baseline"].values

    if np.count_nonzero(valid_baseline) > 0:
        adi, cv = adi_cv(valid_baseline)
    else:
        adi, cv = 2.0, 1.0  # Fallback para séries zeradas

    df["sanitized_demand"] = df["sales_baseline"].copy()

    # Árvore Syntetos & Boylan via métricas calculadas
    if adi < 1.32 and cv < 0.49:
        # Smooth: Média Móvel Local
        window_mean = (
            df.loc[~df["is_stockout"], "sales_baseline"]
            .rolling(window=7, min_periods=1)
            .mean()
        )
        df.loc[df["is_stockout"], "sanitized_demand"] = window_mean

    elif adi >= 1.32 and cv < 0.49:
        # Intermittent: Taxa por Dia de Exposição
        exposure_rate = df.loc[
            ~df["is_stockout"], "sales_baseline"
        ].sum() / max(df["exposure_day"].sum(), 1)
        df.loc[df["is_stockout"], "sanitized_demand"] = exposure_rate

    elif adi < 1.32 and cv >= 0.49:
        # Erratic: Mediana Histórica de Vendas Positivas
        positive_sales = df.loc[
            (df["sales_baseline"] > 0) & (~df["is_stockout"]), "sales_baseline"
        ]
        median_val = positive_sales.median() if len(positive_sales) > 0 else 0
        df.loc[df["is_stockout"], "sanitized_demand"] = median_val

    else:
        # Lumpy: Mediana Ponderada pela Probabilidade
        positive_sales = df.loc[
            (df["sales_baseline"] > 0) & (~df["is_stockout"]), "sales_baseline"
        ]
        median_val = positive_sales.median() if len(positive_sales) > 0 else 0
        p_demand = (df["sales_baseline"] > 0).mean()
        df.loc[df["is_stockout"], "sanitized_demand"] = median_val * p_demand

    return df

# Classe do Pipeline

In [205]:
import numpy as np
import pandas as pd


class DemandSanitizer:
    """Modular pipeline for retail demand time series sanitation and uncensoring.

    Executes stockout censorship detection, promotion de-lifting, robust anomaly
    filtering with calendar immunization, latent demand imputation via the
    Syntetos & Boylan matrix, and promotional effect re-application.

    Processing Flow:
        1. Identification & Exposure: Maps stockout events (stock_eod == 0) and
           calculates effective shelf exposure days while preserving raw sales.
        2. Promo Lift & Unpromotion: Calculates promotional lift using valid
           in-stock exposure days and extracts the organic baseline sales.
        3. Local Outlier Filtering: Applies a 7-day Hampel filter to organic sales,
           protecting special calendar events and applying dynamic threshold floors.
        4. Latent Demand Imputation: Categorizes demand via ADI/CV metrics and imputes
           unserved demand during stockouts using local rolling averages or global exposure rates.
        5. Promo Re-application & Final Assembly: Re-applies promotional lift to imputed
           stockout days and builds the final sanitized demand target.
    """

    # =========================================================================
    # FALLBACK FUNCTIONS
    # =========================================================================

    @staticmethod
    def default_exposure_day_fallback(df: pd.DataFrame) -> pd.Series:
        """Fallback for exposure days calculation when stock tracking is absent or incomplete.

        Scenarios:
            - Missing 'stock_eod' column in input data.
            - Null values in stock level recordings.

        Fallback Logic:
            Defaults exposure_day to 1.0 (fully exposed) and is_stockout to False
            to avoid false-positive demand imputation.

        Args:
            df (pd.DataFrame): Input DataFrame.

        Returns:
            pd.Series: Binary exposure indicator (1.0 for available, 0.0 for stockout).
        """
        if "stock_eod" not in df.columns:
            return pd.Series(1.0, index=df.index)
        return np.where(df["stock_eod"] == 0, 0.0, 1.0)

    @staticmethod
    def default_promo_lift_fallback(
        avg_normal: float, avg_promo: float, max_allowed_lift: float = 5.0
    ) -> float:
        """Fallback mechanism for promotional lift estimation.

        Scenarios:
            - Insufficient historical promotional or non-promotional data.
            - Baseline sales average below minimum threshold (avg_normal <= 0.1).
            - Negative or zero lift (avg_promo <= avg_normal).
            - Extreme calculated lift values exceeding safety boundaries.

        Fallback Logic:
            Returns 1.0 (no promotional effect) if conditions are unsafe, or caps
            the calculated lift at `max_allowed_lift` (default: 5.0x).

        Args:
            avg_normal (float): Mean sales during non-promotional in-stock days.
            avg_promo (float): Mean sales during promotional in-stock days.
            max_allowed_lift (float, optional): Maximum ceiling for lift. Defaults to 5.0.

        Returns:
            float: Safe promotional lift factor (>= 1.0).
        """
        if (
            pd.notna(avg_normal)
            and avg_normal > 0.1
            and pd.notna(avg_promo)
            and avg_promo > avg_normal
        ):
            return min(avg_promo / avg_normal, max_allowed_lift)
        return 1.0

    @staticmethod
    def compute_outlier_threshold_fallback(
        df: pd.DataFrame, eligible_mask: pd.Series, computed_bound: float
    ) -> float:
        """Fallback threshold calculation for Hampel outlier suppression.

        Scenarios:
            - Intermittent time series where median/MAD collapse to zero.
            - Low-volume products where normal sales (e.g., 3 units) get flagged as outliers.

        Fallback Logic:
            Enforces a lower bound on the outlier threshold using the 85th percentile
            of eligible baseline sales or a hard minimum threshold of 3.0 units.

        Args:
            df (pd.DataFrame): Working DataFrame containing 'sales_baseline'.
            eligible_mask (pd.Series): Boolean mask of eligible non-event, in-stock days.
            computed_bound (float): Statistical upper bound derived from Hampel/interval filter.

        Returns:
            float: Adjusted upper bound ensuring low valid sales are not truncated.
        """
        q85 = df.loc[eligible_mask, "sales_baseline"].quantile(0.85)
        min_absolute_threshold = max(q85 if pd.notna(q85) else 0.0, 3.0)
        return max(computed_bound, min_absolute_threshold)

    @staticmethod
    def compute_global_sales_rate_fallback(
        df: pd.DataFrame, valid_exposure_mask: pd.Series
    ) -> float:
        """Fallback rate calculation for latent demand imputation.

        Scenarios:
            - Rolling local window contains only zero-sale days prior to stockout.
            - High-intermittency series (Lumpy/Erratic/Intermittent) according to Syntetos-Boylan.
            - Insufficient rolling window context at the beginning of the series.

        Fallback Logic:
            Computes the item's global daily sales velocity across all valid in-stock
            exposure days (total_valid_sales / total_exposure_days). Returns 0.1 if no sales exist.

        Args:
            df (pd.DataFrame): Working DataFrame containing sales and exposure metrics.
            valid_exposure_mask (pd.Series): Boolean mask indicating days with stock availability.

        Returns:
            float: Global daily sales rate to be used as baseline imputation.
        """
        total_valid_sales = df.loc[valid_exposure_mask, "sales_baseline"].sum()
        total_exposure_days = max(
            df.loc[valid_exposure_mask, "exposure_day"].sum(), 1.0
        )
        rate = total_valid_sales / total_exposure_days
        return rate if rate > 0 else 0.1

    @staticmethod
    def apply_stockout_floor_fallback(
        sanitized: pd.Series, df: pd.DataFrame, global_rate: float
    ) -> pd.Series:
        """Fallback floor enforcer for sanitized demand on stockout days.

        Scenarios:
            - Imputation algorithms output zero for a stockout day.
            - Partial stockouts where raw sales recorded exceed the imputed latent demand.

        Fallback Logic:
            1. Replaces 0.0 imputed values on stockout days with the global sales rate.
            2. Applies an absolute floor using `sales_raw` so sanitized demand is never
               lower than actual observed physical sales.

        Args:
            sanitized (pd.Series): Imputed demand series before final flooring.
            df (pd.DataFrame): Working DataFrame containing 'is_stockout' and 'sales_raw'.
            global_rate (float): Item global sales velocity rate.

        Returns:
            pd.Series: Fully floored and validated sanitized demand series.
        """
        sanitized = sanitized.copy()
        stockout_mask = df["is_stockout"]

        if stockout_mask.any():
            zero_stockouts = stockout_mask & (sanitized == 0.0)
            sanitized.loc[zero_stockouts] = global_rate
            sanitized = np.maximum(sanitized, df["sales_raw"].astype(float))

        return sanitized

    # =========================================================================
    # PIPELINE Core METHODS
    # =========================================================================

    @classmethod
    def identify_stockout(cls, df: pd.DataFrame) -> pd.DataFrame:
        """Identifies stockout dates and computes effective exposure days.

        Scenarios Handled:
            - Standard operational tracking (stock_eod == 0 triggers stockout).
            - Missing raw sales column (creates 'sales_raw' backup).
            - Fallback execution if stock tracking columns are unpopulated.

        Args:
            df (pd.DataFrame): Input raw sales DataFrame.

        Returns:
            pd.DataFrame: DataFrame updated with 'sales_raw', 'is_stockout', and 'exposure_day'.
        """
        df = df.copy()
        if "sales_raw" not in df.columns:
            df["sales_raw"] = df["sales"].copy()

        df["exposure_day"] = cls.default_exposure_day_fallback(df)
        df["is_stockout"] = df["exposure_day"] == 0.0
        return df

    @classmethod
    def extract_promo_baseline(cls, df: pd.DataFrame) -> tuple[pd.DataFrame, float]:
        """Extracts promotional lift and converts sales into organic baseline sales.

        Scenarios Handled:
            - Active promotions with valid non-promo baselines (calculates promo_lift).
            - Missing promo periods, invalid baselines, or excessive lifts (triggers fallback).

        Args:
            df (pd.DataFrame): DataFrame with stockout identification.

        Returns:
            tuple[pd.DataFrame, float]: Transformed DataFrame with 'sales_baseline' and float promo_lift factor.
        """
        df = df.copy()
        valid_days = ~df["is_stockout"]

        avg_normal = df.loc[valid_days & (df["is_promo"] == 0), "sales"].mean()
        avg_promo = df.loc[valid_days & (df["is_promo"] == 1), "sales"].mean()

        promo_lift = cls.default_promo_lift_fallback(avg_normal, avg_promo)

        df["sales_baseline"] = df["sales"].astype(float)
        promo_mask = df["is_promo"] == 1
        df.loc[promo_mask, "sales_baseline"] = (
            df.loc[promo_mask, "sales"] / promo_lift
        )

        return df, promo_lift

    @classmethod
    def filter_organic_outliers(
        cls, df: pd.DataFrame, window_size: int = 7, factor: float = 3.0
    ) -> pd.DataFrame:
        """Filters organic demand spikes using Hampel filter with event protection.

        Scenarios Handled:
            - Spikes on regular non-promo days (truncated to upper bound).
            - Special calendar events / holidays (immunized from filtering).
            - Intermittent demand series where MAD=0 (triggers outlier threshold fallback).
            - Series shorter than window size (bypasses filtering safely).

        Args:
            df (pd.DataFrame): DataFrame containing 'sales_baseline'.
            window_size (int, optional): Rolling window size. Defaults to 7.
            factor (float, optional): MAD multiplier for outlier bounds. Defaults to 3.0.

        Returns:
            pd.DataFrame: DataFrame with 'sales_no_outliers' column.
        """
        df = df.copy()

        if "is_special_event" not in df.columns:
            df["is_special_event"] = False

        eligible_mask = (
            (~df["is_stockout"])
            & (df["is_promo"] == 0)
            & (~df["is_special_event"])
        )

        df["sales_no_outliers"] = df["sales_baseline"].copy()
        valid_baseline = df.loc[eligible_mask, "sales_baseline"]

        if len(valid_baseline) >= window_size:
            is_outlier = hampel_filter(
                df["sales_baseline"], window_size=window_size, factor=factor
            )

            clean_sample = (
                df.loc[eligible_mask & (~is_outlier), "sales_baseline"]
                .dropna()
                .values
            )

            if len(clean_sample) > 0:
                _, upper_bound = StatisticalInterval.compute_interval(
                    clean_sample, method="auto"
                )

                upper_bound = cls.compute_outlier_threshold_fallback(
                    df, eligible_mask, upper_bound
                )

                cut_mask = (
                    eligible_mask
                    & is_outlier
                    & (df["sales_baseline"] > upper_bound)
                )
                df.loc[cut_mask, "sales_no_outliers"] = upper_bound

        return df

    @classmethod
    def _impute_syntetos_boylan_baseline(
        cls, df: pd.DataFrame, adi: float, cv: float
    ) -> pd.Series:
        """Imputes latent baseline demand on stockout days based on Syntetos & Boylan categorization.

        Scenarios Handled:
            - Smooth Demand (ADI < 1.32 & CV < 0.49): Imputes via 7-day rolling mean.
            - Intermittent / Erratic / Lumpy Demand: Imputes via global exposure rate.
            - Rolling mean returning 0.0 or NaNs: Triggers global rate fallback.

        Args:
            df (pd.DataFrame): Working DataFrame.
            adi (float): Average Demand Interval.
            cv (float): Coefficient of Variation.

        Returns:
            pd.Series: Imputed baseline series.
        """
        sanitized_demand = df["sales_no_outliers"].copy()
        is_stockout_mask = df["is_stockout"]
        valid_exposure_mask = ~is_stockout_mask

        global_rate = cls.compute_global_sales_rate_fallback(
            df, valid_exposure_mask
        )

        # Smooth Demand Category
        if adi < 1.32 and cv < 0.49:
            clean_series = df["sales_no_outliers"].where(
                valid_exposure_mask, np.nan
            )

            imputed = (
                clean_series.rolling(window=7, min_periods=1)
                .mean()
                .bfill()
                .ffill()
            )

            imputed = imputed.replace(0.0, global_rate).fillna(global_rate)
            sanitized_demand.loc[is_stockout_mask] = imputed.loc[
                is_stockout_mask
            ]

        # Intermittent / Erratic / Lumpy Demand Category
        else:
            sanitized_demand.loc[is_stockout_mask] = global_rate

        return sanitized_demand

    @staticmethod
    def _reapply_promo_lift_on_stockouts(
        sanitized_demand: pd.Series, df: pd.DataFrame, promo_lift: float
    ) -> pd.Series:
        """Re-applies promotional lift to stockout days that had marketing actions.

        Scenarios Handled:
            - Stockout on a promotional day (is_promo == 1): Multiplies imputed baseline by promo_lift.
            - Stockout on a regular day (is_promo == 0): Retains imputed organic baseline.

        Args:
            sanitized_demand (pd.Series): Imputed baseline demand series.
            df (pd.DataFrame): Working DataFrame with 'is_stockout' and 'is_promo'.
            promo_lift (float): Promotional multiplier factor.

        Returns:
            pd.Series: Demand series with promotional lift restored on stockouts.
        """
        sanitized_demand = sanitized_demand.copy()
        promo_stockout_mask = df["is_stockout"] & (df["is_promo"] == 1)

        if promo_lift > 1.0 and promo_stockout_mask.any():
            sanitized_demand.loc[promo_stockout_mask] *= promo_lift

        return sanitized_demand

    @classmethod
    def impute_latent_demand(
        cls, df: pd.DataFrame, promo_lift: float
    ) -> pd.DataFrame:
        """Orchestrates latent demand imputation across stockouts and reapplies marketing lifts.

        Scenarios Handled:
            - Complete pipeline imputation orchestration.
            - Empty or all-zero sales history (defaults ADI/CV to intermittent).
            - Stockout floor fallback enforcement.

        Args:
            df (pd.DataFrame): Cleaned DataFrame post-outlier filtering.
            promo_lift (float): Calculated promotional lift.

        Returns:
            pd.DataFrame: Output DataFrame with consolidated 'sanitized_demand'.
        """
        df = df.copy()

        clean_history = df.loc[~df["is_stockout"], "sales_no_outliers"].values
        adi, cv = (
            adi_cv(clean_history)
            if np.count_nonzero(clean_history) > 0
            else (2.0, 1.0)
        )

        base_imputed_demand = cls._impute_syntetos_boylan_baseline(
            df, adi, cv
        )
        sanitized = cls._reapply_promo_lift_on_stockouts(
            base_imputed_demand, df, promo_lift
        )

        valid_exposure_mask = ~df["is_stockout"]
        global_rate = cls.compute_global_sales_rate_fallback(
            df, valid_exposure_mask
        )

        df["sanitized_demand"] = cls.apply_stockout_floor_fallback(
            sanitized, df, global_rate
        )
        return df

    @classmethod
    def run_pipeline(
        cls, df: pd.DataFrame, window_size: int = 7, factor: float = 3.0
    ) -> pd.DataFrame:
        """Executes the complete retail demand sanitation and uncensoring pipeline.

        Args:
            df (pd.DataFrame): Raw input DataFrame containing sales, stock, and promo flags.
            window_size (int, optional): Hampel rolling window size. Defaults to 7.
            factor (float, optional): Hampel MAD factor threshold. Defaults to 3.0.

        Returns:
            pd.DataFrame: Fully enriched DataFrame with audit columns and 'sanitized_demand'.
        """
        df_step1 = cls.identify_stockout(df)
        df_step2, promo_lift = cls.extract_promo_baseline(df_step1)
        df_step3 = cls.filter_organic_outliers(
            df_step2, window_size=window_size, factor=factor
        )
        df_final = cls.impute_latent_demand(df_step3, promo_lift=promo_lift)

        return df_final

In [204]:
DemandSanitizer.run_pipeline(df)

,date,sales,stock_eod,is_promo,price,is_stockout,exposure_day,sales_no_outliers,sales_baseline,sanitized_demand,sales_raw,is_special_event
0,2026-01-01,0,10,0,10,False,1.0,0.0,0.0,0.000000,0,False
1,2026-01-02,2,8,0,10,False,1.0,2.0,2.0,2.000000,2,False
2,2026-01-03,0,8,0,10,False,1.0,0.0,0.0,0.000000,0,False
3,2026-01-04,1,7,0,10,False,1.0,1.0,1.0,1.000000,1,False
4,2026-01-05,0,0,0,10,True,0.0,0.0,0.0,0.692308,0,False
5,2026-01-06,15,0,1,5,True,0.0,3.0,3.0,15.000000,15,False
6,2026-01-07,0,0,0,10,True,0.0,0.0,0.0,0.692308,0,False
7,2026-01-08,0,5,0,10,False,1.0,0.0,0.0,0.000000,0,False
8,2026-01-09,0,5,0,10,False,1.0,0.0,0.0,0.000000,0,False
9,2026-01-10,0,0,0,10,True,0.0,0.0,0.0,0.692308,0,False


In [194]:
df

,date,sales,stock_eod,is_promo,price,is_stockout,exposure_day,sales_no_outliers,sales_baseline,sanitized_demand
0,2026-01-01,0,10,0,10,False,1,0.0,0.000,0.000000
1,2026-01-02,2,8,0,10,False,1,2.0,2.000,2.000000
2,2026-01-03,0,8,0,10,False,1,0.0,0.000,0.000000
3,2026-01-04,1,7,0,10,False,1,1.0,1.000,1.000000
4,2026-01-05,0,0,0,10,True,0,0.0,0.000,0.673077
5,2026-01-06,15,0,1,5,True,0,15.0,2.625,0.673077
6,2026-01-07,0,0,0,10,True,0,0.0,0.000,0.673077
7,2026-01-08,0,5,0,10,False,1,0.0,0.000,0.000000
8,2026-01-09,0,5,0,10,False,1,0.0,0.000,0.000000
9,2026-01-10,0,0,0,10,True,0,0.0,0.000,0.673077
